# Aprendizado de Máquina — Lista prática 04

## Métodos Não Paramétricos (KNN)

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Esta lista fecha os três blocos da aula pelo lado prático: a **base de
*splines*** da Seção 2 das notas, o **KNN** da Seção 3 e a leitura que unifica os
dois — **suavizadores lineares** — da Seção 4. O último exercício tem uma
reviravolta:

> **o KNN é um suavizador linear, mas o atalho do LOOCV da Aula 03 não vale para
> ele. Ser linear não basta.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots

import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.neighbors import KNeighborsRegressor

import warnings
warnings.filterwarnings("ignore")

---
## 2. A amostra

A de sempre: $r(x)=\operatorname{sen}(1{,}5x)+0{,}3x$ com ruído $N(0;\,0{,}7^2)$,
$n=50$ pontos, semente 2026. As dobras da validação cruzada também ficam aqui,
para que todos os exercícios comparem os métodos **nas mesmas divisões**.

In [ ]:
def r(x):
    return np.sin(1.5 * x) + 0.3 * x


A, B, SIGMA, N_TR = -3.0, 3.0, 0.7, 50

rng = np.random.default_rng(2026)
x_tr = rng.uniform(A, B, size=N_TR)
y_tr = r(x_tr) + rng.normal(0, SIGMA, size=N_TR)
grade = np.linspace(A, B, 300)

X_tr = x_tr.reshape(-1, 1)
cv = skm.KFold(5, shuffle=True, random_state=2026)

print(f"amostra: n = {N_TR}, x em [{A}, {B}], sigma = {SIGMA}")

---
## Exercício 1 — cada nó compra um grau de liberdade

As notas afirmam duas coisas sobre a base truncada
$1,\,x,\dots,x^k,\,(x-t_j)_+^k$: que ela usa $I = k+1+m$ parâmetros com $m$ nós, e
que a emenda nos nós **sai suave de graça**. Vamos conferir as duas contra a
alternativa ingênua: ajustar uma cúbica **independente** em cada pedaço, sem
restrição nenhuma nas emendas.

In [ ]:
def base_truncada(x, nos, grau=3):
    colunas = [x ** j for j in range(1, grau + 1)]
    colunas += [...]                                        # (a) a parte truncada
    return np.column_stack(colunas)


def por_partes(x, nos, grau=3):
    """Uma cubica INDEPENDENTE em cada pedaco: nada amarra as emendas."""
    bordas = np.r_[-np.inf, nos, np.inf]
    colunas = []
    for t in range(len(bordas) - 1):
        dentro = ((x >= bordas[t]) & (x < bordas[t + 1])).astype(float)
        colunas += [dentro * (x ** j) for j in range(grau + 1)]
    return np.column_stack(colunas)


nos = np.array([-1.5, 0.0, 1.5])
Zs, Zp = base_truncada(x_tr, nos), por_partes(x_tr, nos)

spline = skl.LinearRegression().fit(Zs, y_tr)
pp = skl.LinearRegression(fit_intercept=False).fit(Zp, y_tr)

print(f"spline     : {Zs.shape[1] + 1:2d} parametros   (as notas dizem grau+1+m = {...})")   # (b)
print(f"por partes : {Zp.shape[1]:2d} parametros   (uma cubica por pedaco: 4(m+1) = {...})")  # (c)

In [ ]:
def deriv(f, x0, ordem, lado, h=1e-3):
    """Derivada de ordem `ordem` em x0, usando pontos de um lado so."""
    xs = x0 + lado * h * np.arange(ordem + 1)
    v = f(xs)
    for _ in range(ordem):
        v = np.diff(v) / (lado * h)
    return v[0]


def salto(f, t, ordem):
    """Quanto a derivada de ordem `ordem` pula ao atravessar o no t."""
    return abs(...)                                         # (a)


f_spline = lambda x: spline.predict(base_truncada(x, nos))
f_pp = lambda x: pp.predict(por_partes(x, nos))

print(f"{'no':>6} {'ordem':>6} {'spline':>12} {'por partes':>14}")
for t in nos:
    for o in (0, 1, 2):
        print(f"{t:>6.1f} {o:>6} {salto(f_spline, t, o):>12.2e} {salto(f_pp, t, o):>14.2e}")

print(f"\nEQM contra r:  spline {np.mean((f_spline(grade) - r(grade)) ** 2):.4f}"
      f"   por partes {np.mean((f_pp(grade) - r(grade)) ** 2):.4f}")

---
## Exercício 2 — KNN: o $k$ por validação cruzada

Do outro lado da aula está o KNN, que não monta base nenhuma: guarda os dados e
faz média local. Escolha o $k$ com as dobras da Seção 2 e meça contra a $r$
verdadeira — que só nós conhecemos, e a validação cruzada não.

In [ ]:
ks = np.arange(1, 26)

eqm_knn = np.array([
    -skm.cross_val_score(KNeighborsRegressor(n_neighbors=...),   # (a)
                         X_tr, y_tr, cv=cv,
                         scoring="neg_mean_squared_error").mean()
    for k in ks
])

k_melhor = ks[...]                         # (b)
print(f"melhor k por CV: {k_melhor}  (EQM {eqm_knn.min():.4f})")

knn = KNeighborsRegressor(n_neighbors=k_melhor).fit(X_tr, y_tr)
eqm_r = np.mean((knn.predict(grade.reshape(-1, 1)) - r(grade)) ** 2)
print(f"EQM contra r verdadeira: {eqm_r:.4f}")

---
## Exercício 3 — o KNN é um suavizador linear (e o que isso *não* garante)

A Seção 4 das notas diz que o KNN se escreve como $\widehat r(x)=\sum_i
\ell_i(x)\,y_i$, com pesos que não dependem de $\mathbf{y}$. Avaliado nos próprios
pontos de treino isso vira $\widehat{\mathbf{y}} = \mathbf{H}\mathbf{y}$, com
$H_{ij} = 1/k$ se $j$ é um dos $k$ vizinhos de $i$, e $0$ caso contrário.

Monte essa matriz e confira as duas afirmações da Lista Teórica: que
$h_{ii}=1/k$, e portanto $\operatorname{tr}(\mathbf{H}) = n/k$.

In [ ]:
k = 5
knn5 = KNeighborsRegressor(n_neighbors=k).fit(X_tr, y_tr)
_, viz = knn5.kneighbors(X_tr)        # indices dos k vizinhos de cada ponto de treino

H = np.zeros((N_TR, N_TR))
for i, v in enumerate(viz):
    H[i, v] = ...                     # (a)

print(f"H @ y bate com predict? maior diferenca: {np.abs(H @ y_tr - knn5.predict(X_tr)).max():.2e}")
print(f"diagonal de H: todos iguais a 1/k = {1 / k}? {np.allclose(np.diag(H), 1 / k)}")
print(f"tr(H) = {np.trace(H):.4f}     n/k = {N_TR / k:.4f}")

Agora a parte que interessa. A Aula 03 provou que, para um ajuste linear,
o LOOCV sai de um **único** ajuste:

$$\widehat R_{\text{LOO}} = \frac{1}{n}\sum_{i=1}^n
  \left(\frac{y_i - \widehat r(x_i)}{1 - h_{ii}}\right)^2 .$$

O KNN é um suavizador linear e tem $h_{ii}$ bem definido. Então o atalho vale?
Não responda — meça, comparando com a força bruta ($n$ reajustes).

In [ ]:
def loocv_atalho(k):
    m = KNeighborsRegressor(n_neighbors=k).fit(X_tr, y_tr)
    _, v = m.kneighbors(X_tr)
    Hk = np.zeros((N_TR, N_TR))
    for i, vi in enumerate(v):
        Hk[i, vi] = 1.0 / k
    res = y_tr - Hk @ y_tr
    return np.mean((...) ** 2)                        # (a) aplique a formula


def loocv_bruto(k):
    erros = []
    for i in range(N_TR):
        m = KNeighborsRegressor(n_neighbors=k).fit(np.delete(X_tr, i, axis=0),
                                                   np.delete(y_tr, i))
        erros.append((y_tr[i] - m.predict(X_tr[i:i + 1])[0]) ** 2)
    return np.mean(erros)


print(f"{'k':>3} {'atalho':>10} {'forca bruta':>13} {'razao':>8}")
for k_ in (2, 5, 10):
    a, b = loocv_atalho(k_), loocv_bruto(k_)
    print(f"{k_:>3} {a:>10.4f} {b:>13.4f} {a / b:>8.3f}")

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | 3 nós custam 7 parâmetros no *spline* e 16 na cúbica por partes: 3 restrições por nó |
| 1 | os saltos de $g$, $g'$, $g''$ nos nós são $10^{-9}$ a $10^{-3}$ no *spline* e até $160$ na cúbica por partes |
| 1 | a cúbica por partes ajusta melhor o treino (0,4111 × 0,5444) e é 16× pior contra $r$ (3,5712 × 0,2233) |
| 2 | a CV escolhe $k=5$; contra $r$ o KNN dá 0,1236, melhor que o *spline* de nós chutados |
| 3 | $\operatorname{tr}(\mathbf{H}) = n/k = 10$ para $k=5$, e $\mathbf{H}\mathbf{y}$ reproduz o `predict` a $10^{-16}$ |
| 3 | o atalho do LOOCV **erra** para o KNN: 27% em $k=2$, e para os dois lados conforme $k$ |

**A seguir.** A Aula 05 explica com teoria por que todo método de vizinhança
degrada quando $p$ cresce — e por que, em dimensão alta, a própria noção de
"os $k$ mais próximos" começa a perder sentido.